# Возможности `zemi.toml`

Ноутбук демонстрирует расширения `zemi.toml` относительно стандартного `tomllib` на расположенной рядом конфигурации `test_zemi_toml.toml`.

In [1]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / ".zemicomp").is_file() and (directory / "zemi").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zemi import env, toml

env.path.tmp.mkdir(parents=True, exist_ok=True)
TEST_DIR = PROJECT_ROOT / "tests/zemi_toml"
CONFIG_PATH = TEST_DIR / "test_zemi_toml.toml"
config = toml.load(CONFIG_PATH)
type(config), type(config.arsenal), type(config.arsenal.llamas), len(config.arsenal.llamas)

(zemi.toml.Table, zemi.toml.NamedArray, 2)

## Таблицы и доступ через точку

Обычный `tomllib` возвращает вложенные `dict`. `zemi.toml` возвращает `Table` — подкласс `dict`, поэтому оба способа доступа равноправны.

In [2]:
assert config.arsenal is config["arsenal"]
assert config.arsenal.llamas is config["arsenal"]["llamas"]
assert config.arsenal.llamas["primary"].host == config["arsenal"]["llamas"]["primary"]["host"]

config.arsenal.llamas["primary"].host

'127.0.0.1'

## Именованные массивы таблиц

Массивы `llamas`, `models` и `sessions` преобразованы в `NamedArray`. Элемент доступен по имени, положительному или отрицательному индексу. Имена можно использовать и через точку, если они являются допустимыми Python-идентификаторами.

In [ ]:
primary_by_name = config.arsenal.llamas["primary"]
primary_by_index = config.arsenal.llamas[0]
primary_by_dot = config.arsenal.llamas.primary

assert primary_by_name is primary_by_index is primary_by_dot
assert config.arsenal.llamas[-1] is config.arsenal.llamas["secondary"]

[llama.name for llama in config.arsenal.llamas.values()]

## Рекурсивное сочетание способов доступа

На каждом уровне можно независимо выбирать доступ через точку, имя или индекс.

In [ ]:
session_by_name = config.arsenal.llamas["primary"].models["qwen"].sessions["json_converter"]
session_by_index = config.arsenal.llamas[0].models[0].sessions[1]
session_by_dot = config.arsenal.llamas.primary.models.qwen.sessions.json_converter

assert session_by_name is session_by_index is session_by_dot
session_by_name.name

## Ссылки на файлы `@comp/...` и `@inst/...`

Строковое значение, начинающееся с `@comp/` или `@inst/` и указывающее на файл `.md` или `.txt`, заменяется текстом UTF-8-файла. Обход рекурсивный: ссылки раскрываются и внутри таблиц, и внутри массивов. Пути с другими расширениями остаются исходными строками. В примере `prefix` ссылается на Markdown-файл относительно корня компонента.

In [ ]:
prefix = session_by_name.prefix
source_text = (TEST_DIR / "prefixes/qwen-json.md").read_text(encoding="utf-8")

assert prefix == source_text
assert not prefix.startswith("@comp/")
assert config.non_text_reference == "@comp/runme.toml"
assert config.text_reference == "Текст из TXT-файла.\n"
prefix[:500]

`@comp/` разрешается через `zemi.env.path.comp`, а `@inst/` — через `zemi.env.path.inst`. Расширения `.md` и `.txt` сопоставляются без учёта регистра. Если поддерживаемый текстовый файл отсутствует, `load()` передаёт исключение `FileNotFoundError`; текст читается строго как UTF-8.

In [ ]:
env.path.comp, env.path.inst

## Элемент массива без `name`

Поле `name` необязательно. Безымянный элемент доступен по индексу, но ключа для доступа по имени у него нет. Пустая строка обрабатывается так же, как отсутствие `name`.

In [ ]:
unnamed_toml = '''
[[items]]
value = "без имени"

[[items]]
name = "named"
value = "с именем"
'''

with TemporaryDirectory(dir=env.path.tmp) as directory:
    path = Path(directory) / "unnamed.toml"
    path.write_text(unnamed_toml, encoding="utf-8")
    items = toml.load(path).items

assert items[0].value == "без имени"
assert items[1] is items["named"]
items[0], items["named"]

## Уникальность имён

Непустые `name` должны быть уникальны только в пределах своего массива. Повторение имени приводит к `ValueError`, в тексте которого указан путь проблемного массива.

In [ ]:
duplicate_toml = '''
[[items]]
name = "duplicate"

[[items]]
name = "duplicate"
'''

with TemporaryDirectory(dir=env.path.tmp) as directory:
    path = Path(directory) / "duplicate.toml"
    path.write_text(duplicate_toml, encoding="utf-8")
    try:
        toml.load(path)
    except ValueError as error:
        result = type(error).__name__, str(error)
    else:
        raise AssertionError("Дубликат name не был обнаружен")

result

## Обычные массивы и TOML-типы

Только непустые массивы таблиц становятся `NamedArray`. Массивы строк, чисел и других обычных значений, а также пустые массивы остаются `list`. Числа, даты, булевы значения и другие TOML-типы сохраняют поведение `tomllib`.

In [ ]:
types_toml = '''
ports = [8080, 8081]
empty = []
enabled = true
created = 2026-08-02
'''

with TemporaryDirectory(dir=env.path.tmp) as directory:
    path = Path(directory) / "types.toml"
    path.write_text(types_toml, encoding="utf-8")
    values = toml.load(path)

type(values.ports), values.ports, type(values.empty), values.enabled, type(values.created)

## Итог

| Возможность | `tomllib` | `zemi.toml` |
|---|---|---|
| Загрузка файла по пути | Требуется открыть файл в binary mode | `toml.load(path)` |
| `@comp/...md`, `@inst/...txt` | Обычные строки | Текст связанных UTF-8-файлов; остальные расширения не раскрываются |
| TOML-таблицы | `dict` | `Table`: ключи и dot-доступ |
| Массивы таблиц | `list[dict]` | `NamedArray`: индекс и уникальное имя |
| Элемент без `name` | Не имеет специального смысла | Доступен только по индексу |
| Дубликаты непустого `name` | Разрешены | `ValueError` |
| Обычные массивы и скаляры | Стандартные Python-типы | Без изменений |